# Trends & Raw SQL

Part of the [PyMAUDE](../) examples — see [`quickstart.ipynb`](../quickstart.ipynb) first if you haven't loaded a database yet.

This notebook covers year-over-year trend analysis and dropping down to raw SQL when the built-in query methods aren't enough.

---
## Contents
1. [Setup](#1-setup)
2. [Trend analysis by year](#2-trends)
3. [Raw SQL](#3-sql)

---
## 1. Setup <a id="1-setup"></a>

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = '../maude.duckdb'
DATA_DIR = '../maude_data'
YEARS    = '2024-2026'

db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')
db.add_years(YEARS, tables=['master', 'device', 'text'], download=False)

In [ ]:
# Running example datasets for the rest of this notebook
thrombectomy_broad = db.search_by_device_names(
    [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy']
)
grouped = db.search_by_device_names({
    'venous_stents':  ['venous stent', 'venous stenting'],
    'thrombectomy':   [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'],
})
print(f'Thrombectomy events: {len(thrombectomy_broad):,}  ·  Grouped events: {len(grouped):,}')

---
## 2. Trend analysis by year <a id="2-trends"></a>

`get_trends_by_year()` counts events per calendar year. When results include a `search_group` column it breaks down by group automatically.

In [ ]:
# Overall year-over-year trend
trends = db.get_trends_by_year(thrombectomy_broad)
print('Rotational thrombectomy — events by year:')
print(trends.to_string(index=False))

In [ ]:
# Per-group trends
group_trends = db.get_trends_by_year(grouped)
print('Events by year and device group:')
print(group_trends.to_string(index=False))

In [ ]:
# Pivot for side-by-side comparison
if 'search_group' in group_trends.columns:
    pivot = (
        group_trends
        .pivot(index='year', columns='search_group', values='event_count')
        .fillna(0).astype(int)
    )
    print(pivot.to_string())

---
## 3. Raw SQL <a id="3-sql"></a>

`db.query()` exposes DuckDB directly. Tables available: `master`, `device`, `text`, `patient`, `device_problem`, `patient_problem`.

In [ ]:
# Event type breakdown across all loaded data
db.query("""
    SELECT
        EVENT_TYPE,
        CASE EVENT_TYPE
            WHEN 'D'  THEN 'Death'
            WHEN 'IN' THEN 'Injury'
            WHEN 'M'  THEN 'Malfunction'
            WHEN 'O'  THEN 'Other'
            ELSE EVENT_TYPE
        END AS label,
        COUNT(*) AS n
    FROM master
    GROUP BY EVENT_TYPE
    ORDER BY n DESC
""")

In [ ]:
# Top 10 manufacturers by adverse event volume (master joined to device)
db.query("""
    SELECT
        d.MANUFACTURER_D_NAME,
        COUNT(DISTINCT m.MDR_REPORT_KEY) AS events
    FROM master m
    JOIN device d USING (MDR_REPORT_KEY)
    GROUP BY d.MANUFACTURER_D_NAME
    ORDER BY events DESC
    LIMIT 10
""")

In [ ]:
# Parameterized query — safe for user-supplied input
db.query(
    "SELECT MDR_REPORT_KEY, BRAND_NAME, DATE_RECEIVED FROM device WHERE DEVICE_REPORT_PRODUCT_CODE = ? LIMIT 10",
    params=['NIQ']
)

In [ ]:
db.close()